In [1]:
from openai import OpenAI
import os
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY"),
)


In [2]:
db = {} 

In [3]:
import tqdm, os
import random
import time
reviews = os.listdir("../human_reviews")
def build_embedding(i): 
    while True:
        try:
            if reviews[i] in db.keys():
                print(f"Embedding for {reviews[i]} already exists, skipping...")
                return db[reviews[i]] 
            else:
                with open(f"../human_reviews/{reviews[i]}", "r") as f:
                    content = f.read()

                embedding = client.embeddings.create(
                    model="google/gemini-embedding-001",
                    input=content,
                    encoding_format="float"
                )
                db[reviews[i]] = embedding.data[0].embedding
            return embedding.data[0].embedding
        except Exception as e:
            print(f"Error embedding {reviews[i]}: {e}. Retrying...")
            time.sleep(random.uniform(1, 10))


In [4]:
db.keys()

dict_keys([])

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

with ThreadPoolExecutor(max_workers=50) as executor:
    futures = {executor.submit(build_embedding, i): i for i in range(len(reviews))}
    results = {}
    for f in tqdm.tqdm(as_completed(futures), total=len(futures)):
        idx = futures[f]
        results[idx] = f.result() 

100%|██████████| 17791/17791 [09:37<00:00, 30.80it/s]


In [6]:
keys = list(db.keys())
values = list(db.values())

In [7]:
import numpy as np

values = np.array(values)

In [13]:
query_embedding = client.embeddings.create( 
    model="google/gemini-embedding-001",
    input="very low score paper",
    encoding_format="float"  
)

In [18]:
keys[(np.array(query_embedding.data[0].embedding) @ values.T).argmax()]

'w73feIekdO.md'

In [19]:
filename = keys[(np.array(query_embedding.data[0].embedding) @ values.T).argsort()[-1]]
with open(f"../human_reviews/{filename}", "r") as f:
    print(f"\nMost relevant review for query:\n{f.read()}")


Most relevant review for query:
# Real-time computer vision on low-end boards via clustering motion vectors

- Decision: Reject
- Scores: 3, 1, 6, 3

## Abstract
In this work, we suggest computer vision methods, specifically for video tracking and map creation from video.
To this end, we utilize motion vectors and clusters, which are computed very efficiently in standard video encoders, usually via dedicated hardware.
We suggest a provably good tracking algorithm for clustering these vectors, by considering them as segments.
For this, we utilize a definition of a \emph{coreset} which is essentially a weighted set of points that approximates the fitting loss for every model, up to a multiplicative factor of $1\pm\varepsilon$.
Our method supports $M$-estimators that are robust to outliers, convex shapes, lines, and hyper-planes.
We demonstrate the empirical contribution of our clustering method for video tracking and map creation from video, by running it on micro-computers (Le-Potato a

In [22]:
with open(f"./human_reviews_embeddings.pkl", "wb") as f:
    import pickle
    pickle.dump(db, f)

In [23]:
!cp ./human_reviews_embeddings.pkl ../new/ 